# Notebook 08 — Primo accesso programmatico: GPT e Gemini via API

**Modulo:** 08
**Tempo stimato:** 2-3 ore

## Cosa costruirai

Al termine di questo notebook avrai:
- Autenticazione sicura con OpenAI e Google, tramite `.env`
- Una prima chiamata reale a GPT (OpenAI SDK) e a Gemini (Google SDK)
- Una funzione `calcola_costo` riutilizzabile — la userai in tutti i notebook successivi che fanno chiamate API
- Un wrapper minimo `chiama_llm(prompt, provider)` che astrae le due chiamate

## Prerequisiti

```bash
pip install openai google-generativeai python-dotenv
```

Serve un file `.env` nella cartella del progetto con:
```
OPENAI_API_KEY=sk-...
GOOGLE_API_KEY=...
```


---
## Cella 0 — Setup: installazione e caricamento delle chiavi

Esegui sempre questa cella per prima. Verifica che entrambe le chiavi
vengano caricate correttamente.

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

OPENAI_KEY = os.getenv("OPENAI_API_KEY")
GOOGLE_KEY = os.getenv("GOOGLE_API_KEY")

def verifica_chiave(nome: str, chiave: str | None) -> None:
    if chiave:
        print(f"OK {nome}: {chiave[:8]}...")
    else:
        print(f"MANCANTE {nome}: controlla il tuo file .env")

verifica_chiave("OPENAI_API_KEY", OPENAI_KEY)
verifica_chiave("GOOGLE_API_KEY", GOOGLE_KEY)


---
## Cella 1 — Prima chiamata a OpenAI

La chiamata più semplice possibile. Osserva:
- Come si inizializza il client
- La struttura del parametro `messages`
- Come estrarre il testo dalla risposta
- Cosa restituisce `risposta.usage`

In [ ]:
from openai import OpenAI

cliente_openai = OpenAI(api_key=OPENAI_KEY)

risposta_openai = cliente_openai.chat.completions.create(
    model="gpt-5-mini",
    max_tokens=256,
    messages=[
        {"role": "system", "content": "Sei un assistente conciso. Rispondi sempre in una frase."},
        {"role": "user", "content": "Cos'è un embedding nel contesto degli LLM?"}
    ]
)

testo = risposta_openai.choices[0].message.content
print("Risposta:", testo)

uso = risposta_openai.usage
print(f"\nToken di input:  {uso.prompt_tokens}")
print(f"Token di output: {uso.completion_tokens}")
print(f"Totale:          {uso.total_tokens}")


---
## Cella 2 — Prima chiamata a Gemini

La stessa domanda, con l'SDK di Google. Nota le differenze:
- Il modello si istanzia con `genai.GenerativeModel(...)`
- La risposta è in `risposta.text`
- I token si leggono da `risposta.usage_metadata`

In [ ]:
import google.generativeai as genai

genai.configure(api_key=GOOGLE_KEY)

modello_gemini = genai.GenerativeModel("gemini-3-flash")

risposta_gemini = modello_gemini.generate_content(
    "Cos'è un embedding nel contesto degli LLM? Rispondi in una frase."
)

print("Risposta:", risposta_gemini.text)

uso_gemini = risposta_gemini.usage_metadata
print(f"\nToken di input:  {uso_gemini.prompt_token_count}")
print(f"Token di output: {uso_gemini.candidates_token_count}")


---
## Cella 3 — Tabella comparativa (promemoria)

| Aspetto | OpenAI | Google (Gemini) |
|---|---|---|
| Istanza del client | `OpenAI(api_key=...)` | `genai.configure(api_key=...)` |
| Prompt di sistema | dentro `messages`, `role: "system"` | parametro separato o parte del prompt |
| Testo della risposta | `risposta.choices[0].message.content` | `risposta.text` |
| Token di input | `risposta.usage.prompt_tokens` | `risposta.usage_metadata.prompt_token_count` |
| Token di output | `risposta.usage.completion_tokens` | `risposta.usage_metadata.candidates_token_count` |

Queste differenze sono esattamente ciò che il wrapper della Cella 6
nasconderà dietro un'unica funzione.

---
## Cella 4 — Funzione `calcola_costo`

Da qui in avanti, in **tutti** i notebook successivi del corso che fanno
chiamate API, riuseremo questa stessa funzione — non la reinventeremo.

**Attenzione:** i prezzi della tabella `PREZZI` sono un **esempio
illustrativo**. Verifica sempre i prezzi aggiornati sul sito del provider
prima di presupuestare un progetto reale.

In [ ]:
PREZZI = {
    "gpt-5-mini":     {"input": 0.25, "output": 2.00},
    "gpt-5":          {"input": 2.50, "output": 10.00},
    "gemini-3-flash": {"input": 0.10, "output": 0.40},
    "gemini-3-pro":   {"input": 2.00, "output": 8.00},
}

def calcola_costo(token_input: int, token_output: int, modello: str) -> dict:
    """Calcola il costo in USD di una chiamata, dato il modello usato.

    Prezzi in PREZZI sono un esempio illustrativo: verifica sempre i valori
    aggiornati sul sito del provider prima di presupuestare un progetto reale.
    """
    if modello not in PREZZI:
        raise ValueError(f"Modello '{modello}' non presente nella tabella prezzi.")

    prezzi = PREZZI[modello]
    costo_input  = (token_input  / 1_000_000) * prezzi["input"]
    costo_output = (token_output / 1_000_000) * prezzi["output"]
    costo_totale = costo_input + costo_output

    return {
        "modello":          modello,
        "token_input":      token_input,
        "token_output":     token_output,
        "costo_input_usd":  round(costo_input, 6),
        "costo_output_usd": round(costo_output, 6),
        "costo_totale_usd": round(costo_totale, 6),
    }


# --- Applichiamola alle due chiamate fatte finora ---
costo_openai = calcola_costo(uso.prompt_tokens, uso.completion_tokens, "gpt-5-mini")
costo_gemini = calcola_costo(uso_gemini.prompt_token_count, uso_gemini.candidates_token_count, "gemini-3-flash")

print("=== Costo chiamata OpenAI ===")
for k, v in costo_openai.items():
    print(f"  {k}: {v}")

print("\n=== Costo chiamata Gemini ===")
for k, v in costo_gemini.items():
    print(f"  {k}: {v}")


---
## Esercizio — TODO: proiezione di costo mensile

Scrivi una funzione `proietta_costo_mensile` che, dati i token medi di
input/output per richiesta, il modello, il numero di richieste al giorno e
il numero di utenti, stimi il costo mensile totale (assumi 30 giorni).

In [ ]:
def proietta_costo_mensile(
    token_input_per_richiesta: int,
    token_output_per_richiesta: int,
    modello: str,
    richieste_al_giorno_per_utente: int,
    numero_utenti: int,
) -> None:
    """Proietta il costo mensile di un'applicazione, riusando calcola_costo."""
    # TODO: calcola richieste_totali_al_giorno = richieste_al_giorno_per_utente * numero_utenti

    # TODO: usa calcola_costo() per ottenere il costo di UNA SINGOLA richiesta

    # TODO: calcola il costo_giornaliero (costo per richiesta * richieste totali al giorno)
    # e il costo_mensile (costo_giornaliero * 30)

    # TODO: stampa modello, utenti, richieste totali/giorno, costo giornaliero e costo mensile
    pass


# Prova la funzione: 500 utenti, 3 richieste al giorno ciascuno, con gpt-5-mini
proietta_costo_mensile(
    token_input_per_richiesta=300,
    token_output_per_richiesta=150,
    modello="gpt-5-mini",
    richieste_al_giorno_per_utente=3,
    numero_utenti=500,
)


---
## Sfida finale — Wrapper minimo `chiama_llm`

Costruisci una funzione che astrae la differenza tra le due chiamate viste
finora. È una versione **volutamente minima**: un solo messaggio per
chiamata, nessuna gestione avanzata di errori. Nei moduli successivi la
estenderemo con ruoli, memoria di conversazione e altri provider.

Completa i frammenti marcati con `# TODO`.

In [ ]:
def chiama_llm(prompt: str, provider: str = "openai") -> dict:
    """
    Wrapper unificato: chiama OpenAI o Gemini a seconda del provider scelto.

    Restituisce un dict con:
      - testo: la risposta del modello
      - token_input / token_output: token consumati
      - modello: il modello effettivamente usato
      - provider: il provider usato
    """
    if provider == "openai":
        risposta = cliente_openai.chat.completions.create(
            model="gpt-5-mini",
            max_tokens=512,
            messages=[{"role": "user", "content": prompt}],
        )
        return {
            "testo":        risposta.choices[0].message.content,
            "token_input":  risposta.usage.prompt_tokens,
            "token_output": risposta.usage.completion_tokens,
            "modello":      "gpt-5-mini",
            "provider":     provider,
        }

    elif provider == "gemini":
        # TODO: istanzia genai.GenerativeModel("gemini-3-flash") e chiama generate_content(prompt)

        # TODO: restituisci un dict con la stessa struttura del ramo "openai" sopra,
        # leggendo token_input/token_output da risposta.usage_metadata
        pass

    else:
        raise ValueError(f"Provider '{provider}' non supportato. Usa 'openai' o 'gemini'.")


# --- Prova il wrapper su entrambi i provider ---
DOMANDA = "Spiega in una frase cos'è il fine-tuning."

for provider in ["openai", "gemini"]:
    risultato = chiama_llm(DOMANDA, provider=provider)
    costo = calcola_costo(risultato["token_input"], risultato["token_output"], risultato["modello"])
    print(f"[{provider.upper()} / {risultato['modello']}]")
    print(f"  Risposta: {risultato['testo']}")
    print(f"  Token:    {risultato['token_input']} input + {risultato['token_output']} output")
    print(f"  Costo:    ${costo['costo_totale_usd']:.6f}")
    print()


### Estensioni opzionali (se finisci prima)

1. **Aggiungi la gestione degli errori:** cosa succede se la chiave API non
   è valida o il servizio non risponde? Aggiungi un blocco `try/except`
   attorno alle chiamate.
2. **Parametro `modello` opzionale:** modifica `chiama_llm` per accettare un
   parametro `modello` che, se fornito, sovrascrive il modello di default per
   quel provider.
3. **Confronta i costi su un caso reale:** usa `proietta_costo_mensile` per
   confrontare `gpt-5-mini` e `gemini-3-flash` con gli stessi parametri.
   Quale risulta più economico per il tuo scenario?

In [ ]:
# Spazio per le estensioni opzionali


---
## Autovalutazione della settimana

Rispondi in questa cella prima di chiudere il notebook:

1. **Cosa ha funzionato bene questa settimana?**
   ...

2. **Cosa non hai capito del tutto?**
   ...

3. **Cosa cambieresti nel tuo approccio la settimana prossima?**
   ...

4. **Tempo reale dedicato:**
   ...

---
*Notebook 08 — Modulo 08 del Corso Completo di IA Generativa*